# Computação Paralela — Aula 2
## Modelos de Programação Paralela em Python

**Fork-Join • SPMD • DAG • Threads • Processos • `Executor`/`Future` • GIL**  
Curso Superior de Tecnologia em Inteligência Artificial — FATESG-GO

### Situação profissional
Uma integradora de IA recebe lotes crescentes de arquivos de biometria/inspeção. Para cada item, o pipeline precisa **ler bytes**, executar um **cálculo de assinatura** e **consolidar/validar** os resultados. A leitura contém espera de I/O; a assinatura é deliberadamente implementada como cálculo puro em Python.

Nesta aula, a pergunta central é:

> **Uma única estratégia de paralelismo serve para leitura e cálculo?**

### Capacidades mobilizadas
- representar dependências por um grafo de tarefas (DAG);
- distinguir **Fork-Join** e **SPMD**;
- classificar etapas como **I/O-bound** ou **CPU-bound**;
- selecionar `ThreadPoolExecutor` ou `ProcessPoolExecutor` de forma justificada;
- implementar com **no máximo 2 workers**;
- validar equivalência com baseline sequencial;
- registrar hipótese, evidência, overhead e limitações do ambiente.

### Ancoragem da aula
Material alinhado ao **Plano de Ensino FO-178 2026/2 — Aula 2**, ao **PPC do CST em Inteligência Artificial — Anexo I**, à **Metodologia SENAI de Educação Profissional (MSEP)** e às referências da unidade curricular. O roteiro da aula orienta dados sintéticos, `max_workers=2`, funções de worker no nível de módulo e validação automática de equivalência.

### Versão do estudante
Este notebook **dá bastante suporte**, mas preserva algumas decisões para você completar. Você encontrará:

- código-base pronto e testável;
- **TODOs pequenos e localizados**;
- dicas logo abaixo dos TODOs;
- validações que avisam quando uma etapa ainda não foi concluída, em vez de quebrar o notebook inteiro.

A ideia é que você tenha energia para **raciocinar sobre modelo, executor, ordem e correção**, e não perca tempo com detalhes periféricos.

## 0. Como usar este notebook — Google Colab e Visual Studio Code

### Google Colab
1. Faça upload do `.ipynb` no Colab ou abra-o pelo Google Drive.
2. Use **Ambiente de execução → Executar tudo** apenas depois de ler os enunciados marcados como **ATIVIDADE**.
3. A prática usa somente a **biblioteca-padrão do Python**; não há `pip install` obrigatório.
4. Para a parte com processos, o notebook cria um arquivo `.py`. No Colab, ele aparecerá no painel **Arquivos** à esquerda.
5. O script é executado pelo próprio interpretador do ambiente usando `subprocess`, o que torna o exemplo mais portátil do que definir workers diretamente em células interativas.

### Visual Studio Code
1. Instale/ative as extensões **Python** e **Jupyter**.
2. Abra o `.ipynb` e selecione um kernel Python 3.
3. Execute as células em ordem.
4. O arquivo `.py` criado na parte de processos aparecerá no **Explorer** da mesma pasta do notebook; você pode abri-lo, editar e salvar normalmente.
5. O notebook chamará esse script com o mesmo Python do kernel (`sys.executable`).

### Regra da aula
Usaremos **`MAX_WORKERS = 2`** nos exemplos dos estudantes. Os tempos são evidências do **seu ambiente e desta carga**, não uma regra universal.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import platform
import statistics
import subprocess
import sys
import sysconfig
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

MAX_WORKERS = 2
DATA_DIR = Path.cwd() / "aula2_dados_sinteticos"

print("Diretório de trabalho:", Path.cwd())
print("MAX_WORKERS:", MAX_WORKERS)

### Diagnóstico antes de decidir
A aula não deve assumir cegamente versão do Python, quantidade de CPUs visíveis ou estado do GIL. Execute a célula e registre a informação relevante para sua conclusão.

In [ ]:
def diagnosticar_ambiente() -> dict:
    cpu_count_fn = getattr(os, "process_cpu_count", os.cpu_count)
    cpu_count = cpu_count_fn() or 1
    supports_ft = sysconfig.get_config_var("Py_GIL_DISABLED") == 1
    gil_check = getattr(sys, "_is_gil_enabled", None)
    gil_enabled = gil_check() if callable(gil_check) else None
    ambiente = "Google Colab" if importlib.util.find_spec("google.colab") else "Jupyter/VS Code ou outro"
    return {
        "ambiente": ambiente,
        "implementacao": platform.python_implementation(),
        "python": platform.python_version(),
        "cpus_logicas_visiveis": cpu_count,
        "build_free_threaded": supports_ft,
        "gil_ativo_detectavel": gil_enabled,
    }

info = diagnosticar_ambiente()
for chave, valor in info.items():
    print(f"{chave}: {valor}")

**ATIVIDADE 0 — hipótese inicial (2 min)**  
Antes dos testes, complete em uma célula Markdown abaixo:

- Para **leitura com espera**, eu espero que `____________` seja mais adequado porque `____________`.
- Para **cálculo puro em Python**, eu espero que `____________` seja mais adequado porque `____________`.
- Minha hipótese depende de: `____________`.

---
# Prática 1 
## Decompor, representar e testar o estágio de I/O

O produto desta parte é: **classificação + DAG + matriz de decisão + mini-experimento com threads**.

## 1. Classifique as etapas do pipeline

Use as categorias **I/O-bound**, **CPU-bound** ou **coordenação/leve**. Algumas linhas já vêm orientadas.

| Etapa | Pista | Sua classificação |
|---|---|---|
| Listar caminhos | percorre diretório | `coordenação/leve` |
| Ler bytes | pode esperar armazenamento/rede | `I/O-bound` |
| Consultar metadados remotos | envolve espera de serviço/rede | `I/O-bound` |
| Calcular assinatura | laços Python e operações inteiras | `CPU-bound` |
| Ordenar resultados finais | conjunto pequeno | `coordenação/leve` |
| Validar equivalência | comparação/assert | `coordenação/leve` |

**Dica:** pergunte se o tempo dominante está em **esperar** um recurso externo ou em **executar instruções** no processador.

## 2. Esboce o DAG

Nós disponíveis:

`listar caminhos` · `ler item` · `calcular assinatura do mesmo item` · `consolidar` · `validar`

Crie abaixo um desenho textual ou Mermaid/Markdown com as dependências.

**Pistas suficientes para não travar:**
1. a assinatura de um item não pode existir antes da leitura **daquele mesmo item**;
2. itens diferentes formam ramos independentes;
3. a consolidação final depende das assinaturas necessárias.

> **TODO:** insira seu DAG em uma nova célula Markdown.

In [ ]:
### DAG textual possível

'''text
listar caminhos ──> ler item 0 ──> assinatura 0 ──┐
   ├── ler item 0 ──> assinatura 0 ──┐
   ├── ler item 1 ──> assinatura 1 ──┤
   ├── ...                           ├──> consolidar ──> validar
  alidar ──> ler item 2 ──> assinatura 2 ──┘`
   └── ler item 7 ──> assinatura 7 ──┘


'''

'text\nlistar caminhos ──> ler item 0 ──> assinatura 0 ──┐\n   ├── ler item 0 ──> assinatura 0 ──┐\n   ├── ler item 1 ──> assinatura 1 ──┤\n   ├── ...                           ├──> consolidar ──> validar\n   validar ──> ler item 2 ──> assinatura 2 ──┘\n   ├── ler item 3 ──> assinatura 3 ──�\n   └── ler item 7 ──> assinatura 7 ──┘\n\n\n'

## 3. Matriz de decisão — registre antes de executar

Complete pelo menos as colunas **modelo**, **executor** e **risco/overhead**.

| Etapa | Workload | Modelo possível | Executor/mecanismo | Evidência que vamos obter | Risco/overhead |
|---|---|---|---|---|---|
| Leitura do lote | I/O-bound | `Fork-Join` | `ThreadPoolExecutor` | tempo + equivalência | `excesso de threads, latência artificial` |
| Assinatura por item | CPU-bound | SPMD pode descrever partições | `ProcessPoolExecutor` | tempo + equivalência | `criação, pickle, transferência, granularidade` |
| Consolidação | leve | Join | sequencial | ordem/correção | baixo |

**Dica:** Fork-Join e SPMD não precisam competir; uma região Fork-Join pode distribuir partições que executam o mesmo programa.

## 4. Dados sintéticos reproduzíveis

Não precisamos de base biométrica real nesta aula. Os arquivos binários simulam um lote e evitam que rede, licença ou privacidade contaminem o experimento.

In [ ]:
def criar_arquivos_sinteticos(diretorio: Path, quantidade: int = 8, tamanho: int = 4096) -> list[Path]:
    diretorio.mkdir(parents=True, exist_ok=True)
    for antigo in diretorio.glob("amostra_*.bin"):
        antigo.unlink()

    caminhos = []
    for indice in range(quantidade):
        dados = bytes((indice * 17 + offset * 31) % 256 for offset in range(tamanho))
        caminho = diretorio / f"amostra_{indice:02d}.bin"
        caminho.write_bytes(dados)
        caminhos.append(caminho)
    return caminhos

caminhos = criar_arquivos_sinteticos(DATA_DIR)
print("Arquivos criados:", len(caminhos))
print("Primeiros:", [p.name for p in caminhos[:3]])
print("Tamanho do primeiro arquivo:", caminhos[0].stat().st_size, "bytes")

In [ ]:
def medir(funcao, repeticoes: int = 3):
    # Executa a função algumas vezes e devolve (último_resultado, mediana_em_segundos).
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = funcao()
        tempos.append(time.perf_counter() - inicio)
    return resultado, statistics.median(tempos)

In [ ]:
def read_sample(path: Path) -> tuple[str, bytes]:
    # Representa uma leitura com espera de armazenamento/rede.
    time.sleep(0.08)
    return path.name, path.read_bytes()


def cpu_signature(item: tuple[str, bytes], rounds: int = 1200) -> tuple[str, int]:
    # Carga CPU-bound pura em Python. Mesma entrada -> mesma saída.
    name, data = item
    acc = 2166136261
    for _ in range(rounds):
        for byte in data[:512]:
            acc ^= byte
            acc = (acc * 16777619) & 0xFFFFFFFF
    return name, acc

## 5. Baseline sequencial de I/O

Primeiro crie o **oráculo**: uma versão simples que usaremos para conferir a versão concorrente.

In [ ]:
io_seq, t_io_seq = medir(lambda: [read_sample(caminho) for caminho in caminhos])

print(f"I/O sequencial - mediana didática: {t_io_seq:.3f} s")
print("Quantidade de resultados:", len(io_seq))
print("Primeiro resultado:", io_seq[0][0], len(io_seq[0][1]), "bytes")

## 6. Mini-experimento com `ThreadPoolExecutor`

Complete **somente a linha marcada**. Todo o restante já está pronto.

**Dica 1:** o executor precisa aplicar `read_sample` a cada elemento de `caminhos`.  
**Dica 2:** nesta primeira versão queremos preservar a ordem das entradas.  
**Dica 3:** o método mais direto do executor para isso começa com `m`.

In [ ]:
io_thread = None
t_io_thread = None

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    inicio = time.perf_counter()
    # TODO: substitua None pela expressão que aplica read_sample aos caminhos
    # Exemplo de forma: list(executor.________(________, ________))
    io_thread = None
    t_io_thread = time.perf_counter() - inicio

if io_thread is None:
    print("⏳ Etapa ainda não concluída: preencha o TODO do ThreadPoolExecutor.")
else:
    assert io_thread == io_seq
    print(f"I/O com threads: {t_io_thread:.3f} s")
    print("✅ Equivalência confirmada com o baseline sequencial.")

### Registro de evidência da Prática 1
Preencha:

- I/O sequencial: `_____ s`
- I/O com 2 threads: `_____ s`
- Os resultados foram equivalentes? `sim / não`
- A melhoria, se ocorreu, veio de **mais CPU** ou de **sobreposição de espera**? `____________`
- Uma limitação desta medição: `____________`

> **Checkpoint:** não conclua “threads são sempre mais rápidas”. Conclua apenas sobre **esta carga e este ambiente**.

---
# Pós-intervalo — decisão robusta
## `map`, `submit`, `Future`, `as_completed`, ordem e GIL

## 7. `map` versus `submit`/`as_completed`

`executor.map(...)` facilita obter resultados na ordem das entradas. Com `submit`, cada chamada produz um `Future`; `as_completed` permite tratar resultados na ordem **em que terminam**.

Vamos forçar durações diferentes para que a diferença fique visível.

In [ ]:
def read_sample_variable(index_and_path: tuple[int, Path]):
    indice, path = index_and_path
    # Itens iniciais esperam mais; itens posteriores podem terminar antes.
    atraso = 0.02 + (7 - indice) * 0.015
    time.sleep(atraso)
    return indice, path.name, len(path.read_bytes()), atraso

pares = list(enumerate(caminhos))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futuros = {executor.submit(read_sample_variable, par): par[0] for par in pares}
    conclusao = [future.result() for future in as_completed(futuros)]

print("Ordem de conclusão:", [r[0] for r in conclusao])
print("Ordem de entrada:   ", list(range(len(caminhos))))

### ATIVIDADE — restaurar a ordem
Crie `conclusao_ordenada` a partir de `conclusao` e confirme que os índices ficam `0,1,2,...,7`.

**Dica:** cada tupla começa com o índice original. Você não precisa reler os arquivos nem executar novas tarefas.

In [ ]:
conclusao_ordenada = None  # TODO: reordene 'conclusao' pelo índice original

if conclusao_ordenada is None:
    print("⏳ Reordene a lista usando o índice que já acompanha cada resultado.")
else:
    assert [r[0] for r in conclusao_ordenada] == list(range(len(caminhos)))
    print("✅ Ordem restaurada:", [r[0] for r in conclusao_ordenada])

## 8. Experimento CPU-bound: baseline e threads

A função `cpu_signature` é deliberadamente **pura em Python**. Use o diagnóstico do ambiente antes de transformar a observação em uma regra geral.

A célula abaixo já está pronta para que você concentre o raciocínio na interpretação.

In [ ]:
# Para esta etapa, usamos os dados do baseline sequencial de leitura.
cpu_seq, t_cpu_seq = medir(lambda: [cpu_signature(item) for item in io_seq])

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    cpu_thread, t_cpu_thread = medir(lambda: list(executor.map(cpu_signature, io_seq)))

assert cpu_seq == cpu_thread
print(f"CPU sequencial: {t_cpu_seq:.3f} s")
print(f"CPU com threads: {t_cpu_thread:.3f} s")
print("✅ Saídas equivalentes.")

**Interprete antes de usar processos:**

- Na sua execução, as threads reduziram claramente o tempo da carga CPU-bound? `____________`
- O diagnóstico mostrou build free-threaded? `____________`
- O estado do GIL pôde ser detectado? `____________`
- Qual hipótese você leva para o teste com processos? `____________`

---
# Prática 2 — 55 min
## Pipeline em dois estágios: threads no I/O + processos na CPU

### Por que usar um arquivo `.py` nesta parte?
O material da aula exige um padrão portátil para `ProcessPoolExecutor`: worker no nível do módulo e criação do pool protegida por `if __name__ == "__main__"`. Ambientes interativos variam; por isso, o notebook cria um **script companheiro** e o executa com o mesmo Python do kernel.

Isso funciona tanto no **Colab** quanto no **VS Code** e evita que a aula vire uma investigação de peculiaridades do kernel Jupyter.

## 9. Script companheiro — complete 3 pontos

A célula abaixo grava `aula2_processos_aluno.py`. Há somente **três TODOs conceituais**:

1. aplicar a leitura com o executor adequado;
2. aplicar a assinatura com o executor adequado;
3. validar que baseline e pipeline produzem exatamente o mesmo resultado.

No Colab, você pode abrir o arquivo no painel **Arquivos**. No VS Code, ele aparecerá no **Explorer**.

In [ ]:
SCRIPT_ALUNO = r'''from __future__ import annotations

import statistics
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

MAX_WORKERS = 2
DATA_DIR = Path.cwd() / "aula2_dados_sinteticos"


def read_sample(path: Path) -> tuple[str, bytes]:
    time.sleep(0.08)
    return path.name, path.read_bytes()


def cpu_signature(item: tuple[str, bytes], rounds: int = 1200) -> tuple[str, int]:
    name, data = item
    acc = 2166136261
    for _ in range(rounds):
        for byte in data[:512]:
            acc ^= byte
            acc = (acc * 16777619) & 0xFFFFFFFF
    return name, acc


def medir(funcao, repeticoes: int = 3):
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = funcao()
        tempos.append(time.perf_counter() - inicio)
    return resultado, statistics.median(tempos)


def main() -> None:
    caminhos = sorted(DATA_DIR.glob("amostra_*.bin"))
    if not caminhos:
        raise RuntimeError("Execute primeiro a célula que cria os dados sintéticos.")

    baseline, t_baseline = medir(
        lambda: [cpu_signature(read_sample(caminho)) for caminho in caminhos]
    )

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # TODO 1: aplique read_sample a todos os caminhos preservando a ordem
        lidos = None

    if lidos is None:
        raise RuntimeError("TODO 1 ainda não preenchido.")

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # TODO 2: aplique cpu_signature aos itens lidos
        paralelo = None

    if paralelo is None:
        raise RuntimeError("TODO 2 ainda não preenchido.")

    # TODO 3: substitua False por uma condição de equivalência
    if not False:
        raise RuntimeError("TODO 3: valide baseline e paralelo.")

    print("Baseline e pipeline são equivalentes.")
    print(f"Baseline completo: {t_baseline:.3f} s (mediana didática)")
    print("Primeiras assinaturas:", paralelo[:3])


if __name__ == "__main__":
    main()
'''

script_path = Path.cwd() / "aula2_processos_aluno.py"
script_path.write_text(SCRIPT_ALUNO, encoding="utf-8")
print("Arquivo criado:", script_path)
print("Abra o arquivo, preencha TODO 1, TODO 2 e TODO 3, salve e depois execute a próxima célula.")

### Dicas para os três TODOs
- **TODO 1:** você já usou a ideia na Prática 1. Use `executor.map(...)` e converta o resultado para lista.
- **TODO 2:** a estrutura é análoga, mas agora o executor é de **processos** e a função é `cpu_signature`.
- **TODO 3:** compare a lista do baseline com a lista produzida pelo pipeline.

Essas dicas entregam a **forma**, mas você ainda precisa escolher corretamente função e entrada em cada estágio.

In [ ]:
script_path = Path.cwd() / "aula2_processos_aluno.py"
texto = script_path.read_text(encoding="utf-8")
pendencias = [marcador for marcador in ("TODO 1", "TODO 2", "TODO 3") if marcador in texto]

# Os comentários TODO podem continuar existindo depois de você preencher as linhas.
# Por isso, tentamos executar e mostramos a mensagem do script se algo ainda estiver incompleto.
resultado = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=Path.cwd(),
    text=True,
    capture_output=True,
)

print(resultado.stdout)
if resultado.returncode != 0:
    print("O script ainda precisa de ajuste ou encontrou um erro:\n")
    print(resultado.stderr)
else:
    print("✅ Script executado com sucesso no Python:", sys.executable)

## 10. SPMD — duas formas de particionar 8 itens em 2 workers

Queremos que cada participante execute **o mesmo programa** sobre dados diferentes. A identidade do item precisa acompanhar o resultado.

### Parte A — distribuição cíclica já pronta
Veja o padrão e depois implemente blocos contíguos.

In [ ]:
nomes = [p.name for p in caminhos]

particoes_ciclicas = [nomes[worker::MAX_WORKERS] for worker in range(MAX_WORKERS)]
for worker, particao in enumerate(particoes_ciclicas):
    print(f"Worker {worker} - cíclica:", particao)

### ATIVIDADE — blocos contíguos
Para 8 itens e 2 workers, o esperado conceitualmente é:

- Worker 0: itens 0–3
- Worker 1: itens 4–7

Implemente sem digitar os nomes manualmente.

**Dica:** calcule `tamanho_bloco = len(nomes) // MAX_WORKERS` e use fatias.

In [ ]:
particoes_blocos = None  # TODO: produza uma lista com 2 listas contíguas

if particoes_blocos is None:
    print("⏳ Construa as duas partições usando slicing.")
else:
    achatada = [item for bloco in particoes_blocos for item in bloco]
    assert achatada == nomes
    assert len(achatada) == len(set(achatada))
    for worker, particao in enumerate(particoes_blocos):
        print(f"Worker {worker} - bloco:", particao)
    print("✅ Cobertura completa e sem sobreposição.")

## 11. Entregável final — justificativa técnica curta

Preencha em **6 a 10 linhas**:

1. Qual foi o modelo de programação predominante no pipeline e onde o SPMD apareceu?
2. Por que threads foram consideradas para o I/O?
3. Por que processos foram considerados para a etapa CPU-bound pura em Python?
4. Que overheads podem reduzir ou eliminar a vantagem dos processos?
5. Como você demonstrou que a versão paralela preserva o resultado?
6. Qual característica do seu ambiente impede generalizar os tempos observados?

> O critério não é escrever “processos são mais rápidos”; é **ligar workload, modelo, executor, evidência e limitação**.

## Checklist de correção

Marque antes de entregar:

- [ ] Registrei hipótese antes do teste.
- [ ] Classifiquei I/O-bound e CPU-bound corretamente.
- [ ] Representei dependências no DAG.
- [ ] Usei `MAX_WORKERS = 2`.
- [ ] Validei threads de I/O contra o baseline.
- [ ] Observei diferença entre ordem de conclusão e ordem de entrada.
- [ ] Executei a etapa CPU com `ProcessPoolExecutor` em arquivo `.py`.
- [ ] Mantive worker no nível do módulo e `if __name__ == "__main__"`.
- [ ] Validei equivalência do pipeline.
- [ ] Expliquei overhead e limites do ambiente.

## Referências da aula

- **SENAI-GO.** Plano de Ensino FO-178 — Computação Paralela, 2026/2. Aula 2: Fork-Join, SPMD, grafos de tarefas, threads, processos, executores, CPU-bound/I/O-bound e GIL.
- **SENAI-GO.** Projeto Pedagógico do Curso Superior de Tecnologia em Inteligência Artificial, 2026. Anexo I — Unidade Curricular Computação Paralela.
- **SENAI/DN.** *Metodologia SENAI de Educação Profissional*. Brasília: SENAI/DN, 2019.
- **BORDIN, Maycon V. et al.** *Processamento Paralelo e Distribuído*. Grupo A, 2021.
- **SILVA, G. P.; BIANCHINI, C. P.; COSTA, E. B.** *Programação paralela e distribuída: com MPI, OpenMP e OpenACC para computação de alto desempenho*. Casa do Código, 2022.
- **TANENBAUM, Andrew S.** *Sistemas Operacionais Modernos*. Pearson, 2009.
- **Python Software Foundation.** Documentação oficial de `concurrent.futures`, processos/threads e GIL, conforme documentação técnica indicada no Plano de Ensino e no roteiro da aula.

> Observação: este notebook não aprofunda locks, semáforos, race conditions, deadlocks, NUMA, MPI, GPU, leis formais de speedup ou profiling avançado, pois esses conteúdos pertencem a outros encontros do plano.